# Data Loader

**Run-Mode**:

The project can run in two modes: sample and full. Please set the mode accordingly in **"run_config.py"**. Sample mode uses a reproducible, contiguous 14-day window with complete weekdays, weekends, and daily demand cycles.

**Requirements**:

To run this notebook in full mode you need an API key from the 'Chicago Data Portal' (https://data.cityofchicago.org/). However, you can skip the API call, if you run in sample mode.

**Description**:

This notebook loads the data sources via API or stores data provided under /raw_data to .parquet files. The .parquet files generate by this notebook are referred to as "bronze" level.

Further this notebooks loads POI data from OpenStreetMap via OSMnx and stores the data as parquet file.

## List of Used Data Sources

**Primary Data Sources**

| Data Source                  | Description                                                       | Link                                                                                                                                                                                                                                                        |
| ---------------------------- | ----------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Taxi Data                    | Chicago Taxi Trips 2024, accessed via API                         | [Taxi Trips 2024](https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data)                                                                                                                                                      |
| Census Tract Data            | Geographic boundaries for Chicago census tracts                   | [Census Tracts](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Census_Tracts/4hp8-2i8z/about_data)                                                                                                                                         |
| Community Areas              | Geographic boundaries for Chicago community areas                 | [Boundaries - Community Areas](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas/igwz-8jzy/about_data)                                                                                                             |                                                                                                                                                            |
| Weather Data - MDW           | Weather observations for Chicago Midway Airport `MDW`             | [MDW ASOS Data Query](https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=MDW&data=all&year1=2024&month1=1&day1=1&year2=2026&month2=5&day2=26&tz=America%2FChicago&format=onlycomma&latlon=yes&elev=yes&missing=null&trace=0.0001&direct=no) |                                                                                                                                       |                                                                                                     |

Link for weather data without query setting: https://mesonet.agron.iastate.edu/request/download.phtml?network=IL_ASOS

Point-of-Interest data was downloaded via OSMnx from OpenStreetMap

Further we use the package "holidays" to get the holidays relevant for chicago.

**Data Pipeline**

```text
SODA API
   ↓
Raw Backup (CSV in full mode, compressed Parquet in sample mode)
   ↓
Parquet
   ↓
Polars LazyFrame & DuckDB
```

**Data Preparation Pipeline**

```text
Raw CSV / Parquet
   ↓
Bronze Parquet
   1:1 from CSV/API, kept as unchanged as possible
   ↓
Silver Parquet
   Cleaned, typed, and filtered data
   ↓
Gold / Features
   Merged and aggregated datasets, ML features, train/test-ready datasets
```

**Pipeline Layers**

| Layer           | Description                                                                  |
| --------------- | ---------------------------------------------------------------------------- |
| Raw data        | Original API export; CSV in full mode and compressed Parquet in sample mode   |
| Bronze Parquet  | One-to-one conversion from CSV/API, kept as unchanged as possible            |
| Silver Parquet  | Cleaned, typed, and filtered version of the data                             |
| Gold / Features | Feature-engineered datasets ready for machine learning and train/test splits |



# Imports and Settings

In [6]:
from __future__ import annotations

from pathlib import Path

import duckdb
import geopandas as gpd
import osmnx as ox
import pandas as pd
import requests

from run_config import (
    PATHS,
    RUN_MODE,
    APP_TOKEN,
    PROJECT_ROOT,
    RAW_DIR,
    PROCESSED_DIR,
    START_DATE,
    END_DATE,
    SAMPLE_START_DATE,
    SAMPLE_END_DATE,
    SAMPLE_MAX_TRIPS,
    SAMPLE_MIN_COMPLETE_DAYS,
)

SAMPLE_MODE = True if RUN_MODE == "sample" else False

if RUN_MODE == "sample":
    print("Runs in SAMPLE_MODE with APP_TOKEN = ", APP_TOKEN)
    
else:
    print("Runs in FULL_MODE with APP_TOKEN = ", APP_TOKEN)

Runs in SAMPLE_MODE with APP_TOKEN =  ***


## Download Taxi Data via API
Load the data from "https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data" using SODA3 API and store it as CSV.

In SAMPLE_MODE it loads the configured contiguous 14-day window in deterministic timestamp order. `SAMPLE_MAX_TRIPS` is only a safety guard: reaching it fails validation because the final day may have been truncated. FULL_MODE keeps the complete configured project period.

If the APP-Token is "***", the API will not run and the configured raw fallback is used.

In [7]:
def validate_taxi_sample(sample_path: Path) -> pd.DataFrame:
    columns = ["trip_start_timestamp", "pickup_community_area"]
    if sample_path.suffix == ".parquet":
        sample = pd.read_parquet(sample_path, columns=columns)
    else:
        sample = pd.read_csv(
            sample_path, usecols=columns, parse_dates=["trip_start_timestamp"]
        )
    sample["trip_start_timestamp"] = pd.to_datetime(sample["trip_start_timestamp"])
    if sample.empty:
        raise ValueError(f"Taxi sample is empty: {sample_path}")

    timestamps = sample["trip_start_timestamp"]
    sample_start = pd.Timestamp(SAMPLE_START_DATE)
    sample_end = pd.Timestamp(SAMPLE_END_DATE)
    expected_dates = pd.date_range(
        sample_start.normalize(), sample_end.normalize(), inclusive="left"
    ).date
    observed_dates = set(timestamps.dt.date.unique())
    missing_dates = sorted(set(expected_dates) - observed_dates)
    rows_per_day = timestamps.dt.date.value_counts().sort_index()
    boundary_hours = sample.assign(
        _date=timestamps.dt.date, _hour=timestamps.dt.hour
    ).groupby("_date")["_hour"].agg(["min", "max"])
    expected_first = expected_dates[0]
    expected_last = expected_dates[-1]

    if len(sample) >= SAMPLE_MAX_TRIPS:
        raise ValueError(
            f"Sample reached the {SAMPLE_MAX_TRIPS:,}-trip safety cap; "
            "raise the cap so that the final day is not truncated."
        )
    if len(observed_dates) < SAMPLE_MIN_COMPLETE_DAYS:
        raise ValueError(
            f"Sample contains only {len(observed_dates)} calendar days; "
            f"at least {SAMPLE_MIN_COMPLETE_DAYS} are required."
        )
    if missing_dates:
        raise ValueError(f"Taxi sample is missing configured dates: {missing_dates}")
    if timestamps.min() < sample_start or timestamps.max() >= sample_end:
        raise ValueError(
            f"Taxi sample lies outside [{SAMPLE_START_DATE}, {SAMPLE_END_DATE})."
        )
    if boundary_hours.loc[expected_first, "min"] != 0:
        raise ValueError("The first sample day does not contain the midnight hour.")
    if boundary_hours.loc[expected_last, "max"] != 23:
        raise ValueError("The last sample day does not contain the 23:00 hour.")

    summary = pd.DataFrame({
        "metric": [
            "trips", "calendar_days", "pickup_community_areas",
            "first_timestamp", "last_timestamp", "min_trips_per_day",
            "max_trips_per_day",
        ],
        "value": [
            f"{len(sample):,}", len(observed_dates),
            sample["pickup_community_area"].nunique(dropna=True),
            timestamps.min(), timestamps.max(),
            int(rows_per_day.min()), int(rows_per_day.max()),
        ],
    })
    print("Validated taxi sample:")
    display(summary)
    return summary


def download_taxi_data_via_api():
    DATASET_ID = "ajtu-isnz"
    API_URL = f"https://data.cityofchicago.org/api/v3/views/{DATASET_ID}/query.csv"

    OUTPUT_PATH = PATHS.raw_taxi_trips
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    if SAMPLE_MODE:
        query = f"""
        SELECT *
        WHERE trip_start_timestamp >= '{SAMPLE_START_DATE}'
          AND trip_start_timestamp < '{SAMPLE_END_DATE}'
        ORDER BY trip_start_timestamp, trip_id
        LIMIT {SAMPLE_MAX_TRIPS}
        """
    else:
        query = f"""
        SELECT *
        WHERE trip_start_timestamp >= '{START_DATE}'
          AND trip_start_timestamp < '{END_DATE}'
        """

    headers = {}

    if APP_TOKEN and APP_TOKEN != "***":
        headers["X-App-Token"] = APP_TOKEN

    payload = {"query": query}

    # Stream CSV to a temporary sibling, then compress sample mode to Parquet.
    temporary_path = OUTPUT_PATH.with_suffix(".download.csv")
    with requests.post(
        API_URL,
        headers=headers,
        json=payload,
        stream=True,
        timeout=180,
    ) as response:
        response.raise_for_status()

        with temporary_path.open("wb") as file_handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file_handle.write(chunk)

    if SAMPLE_MODE:
        validate_taxi_sample(temporary_path)
        duckdb.sql(f"""
            COPY (SELECT * FROM read_csv_auto('{temporary_path}', sample_size=-1))
            TO '{OUTPUT_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """)
        temporary_path.unlink()
    else:
        temporary_path.replace(OUTPUT_PATH)
    print(f"Download finished: {OUTPUT_PATH}")
    print(f"File size: {OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")
    
if APP_TOKEN != "***":
    download_taxi_data_via_api()
else:
    print("No API_TOKEN provided. Runs with raw-data fallback")

if SAMPLE_MODE:
    validate_taxi_sample(PATHS.raw_taxi_trips)

No API_TOKEN provided. Runs with raw-data fallback


## Download POI via OpenStreetMap

In [8]:
PLACE = "Chicago, Illinois, USA"

POI_TAGS: dict[str, dict[str, list[str] | str | bool]] = {
    # Restaurants / Cafés / Bars etc.
    "food_drink": {
        "amenity": [
            "restaurant",
            "cafe",
            "fast_food",
            "bar",
            "pub",
            "food_court",
            "ice_cream",
        ],
    },

    # Rail / CTA / Metra-nahe OSM-Features.
    # Bewusst NICHT public_transport=platform oder bus_stop, sonst wird es sehr viel.
    "train_station": {
        "railway": [
            "station",
            "halt",
            "subway_entrance",
            "tram_stop",
        ],
    },

    # Shops: bewusst eingeschränkt, damit es nicht zu granular wird.
    "shop": {
        "shop": [
            "supermarket",
            "convenience",
            "mall",
            "department_store",
            "clothes",
            "bakery",
            "pharmacy",
            "electronics",
        ],
    },

    # Wahrzeichen / touristisch relevante Orte.
    "landmark": {
        "tourism": [
            "attraction",
            "museum",
            "gallery",
            "viewpoint",
            "zoo",
            "aquarium",
        ],
        "historic": [
            "monument",
            "memorial",
        ],
        "man_made": [
            "tower",
            "lighthouse",
        ],
        "amenity": [
            "theatre",
            "arts_centre",
            "cinema",
        ],
    },
}


def configure_osmnx(project_root: Path) -> None:
    """
    OSMnx-Settings: Cache aktivieren, damit du Overpass nicht unnötig oft belastest.
    """
    cache_dir = project_root / ".cache" / "osmnx"
    cache_dir.mkdir(parents=True, exist_ok=True)

    ox.settings.use_cache = True
    ox.settings.cache_folder = str(cache_dir)
    ox.settings.log_console = True
    ox.settings.requests_timeout = 180


def fetch_poi_category(
    place: str,
    category: str,
    tags: dict[str, list[str] | str | bool],
) -> gpd.GeoDataFrame:
    """
    Lädt eine POI-Kategorie aus OpenStreetMap.
    """
    print(f"Fetching {category} ...")

    gdf = ox.features_from_place(place, tags=tags)

    if gdf.empty:
        return gpd.GeoDataFrame(columns=["poi_category", "geometry"], geometry="geometry", crs="EPSG:4326")

    gdf = gdf.reset_index()
    gdf["poi_category"] = category

    return gdf


def add_point_geometry(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    OSM-Features können Punkte, Polygone oder Linien sein.
    Für ML/Feature Engineering mit Taxi-Daten ist oft ein Punkt pro POI praktischer.
    Daher wird aus jeder Geometrie ein repräsentativer Punkt erzeugt.
    """
    gdf = gdf.copy()

    gdf["geom_type_original"] = gdf.geometry.geom_type

    # Für Flächenberechnung und representative_point besser in ein metrisches CRS projizieren.
    metric_crs = gdf.estimate_utm_crs()
    gdf_metric = gdf.to_crs(metric_crs)

    gdf_metric["area_m2"] = gdf_metric.geometry.area

    # Für Points bleibt representative_point identisch bzw. sinnvoll.
    gdf_metric["geometry"] = gdf_metric.geometry.representative_point()

    gdf_points = gdf_metric.to_crs("EPSG:4326")

    gdf_points["lon"] = gdf_points.geometry.x
    gdf_points["lat"] = gdf_points.geometry.y

    return gdf_points


def clean_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Spalten reduzieren und Namen parquet-/polars-freundlicher machen.
    """
    keep_cols = [
        "element_type",
        "osmid",
        "poi_category",
        "name",
        "amenity",
        "shop",
        "railway",
        "tourism",
        "historic",
        "man_made",
        "brand",
        "operator",
        "opening_hours",
        "addr:housenumber",
        "addr:street",
        "addr:city",
        "website",
        "phone",
        "geom_type_original",
        "area_m2",
        "lat",
        "lon",
        "geometry",
    ]

    existing_cols = [col for col in keep_cols if col in gdf.columns]
    gdf = gdf[existing_cols].copy()

    # Doppelpunkte sind in Geo/OSM üblich, aber für spätere Verarbeitung oft nervig.
    rename_map = {
        col: col.replace(":", "_").replace("-", "_")
        for col in gdf.columns
        if col != "geometry"
    }
    gdf = gdf.rename(columns=rename_map)

    return gdf


def deduplicate_pois(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Ein OSM-Objekt kann in mehreren Kategorien landen.
    Hier werden Duplikate über element_type + osmid entfernt.
    """
    gdf = gdf.copy()

    if "element_type" not in gdf.columns or "osmid" not in gdf.columns:
        return gdf.drop_duplicates()

    gdf["osm_uid"] = gdf["element_type"].astype(str) + "/" + gdf["osmid"].astype(str)

    # Wenn ein POI mehrfach vorkommt, Kategorien zusammenführen.
    rows = []
    for _, group in gdf.groupby("osm_uid", sort=False):
        row = group.iloc[0].copy()
        row["poi_category"] = "|".join(sorted(group["poi_category"].dropna().astype(str).unique()))
        rows.append(row)

    result = gpd.GeoDataFrame(rows, geometry="geometry", crs=gdf.crs)
    return result.drop(columns=["osm_uid"])


def extract_chicago_pois(
    place: str = PLACE,
    poi_tags: dict[str, dict[str, list[str] | str | bool]] = POI_TAGS,
) -> gpd.GeoDataFrame:
    frames: list[gpd.GeoDataFrame] = []

    for category, tags in poi_tags.items():
        gdf_category = fetch_poi_category(place=place, category=category, tags=tags)
        if not gdf_category.empty:
            frames.append(gdf_category)

    if not frames:
        raise ValueError("No POIs found. Check place name or OSM tags.")

    pois = pd.concat(frames, ignore_index=True)
    pois = gpd.GeoDataFrame(pois, geometry="geometry", crs=frames[0].crs)

    pois = deduplicate_pois(pois)
    pois = add_point_geometry(pois)
    pois = clean_columns(pois)

    return pois

In [9]:
project_root = PROJECT_ROOT
configure_osmnx(project_root)

geo_out = PATHS.bronze_osm_geo
tabular_out = PATHS.bronze_osm
geo_out.parent.mkdir(parents=True, exist_ok=True)

pois = extract_chicago_pois()

# GeoParquet: Geometrie bleibt erhalten.
pois.to_parquet(geo_out, index=False)

# Normales Parquet für Polars/ML: lat/lon bleiben, geometry wird entfernt.
pois.drop(columns="geometry").to_parquet(tabular_out, index=False)

print(f"Saved GeoParquet: {geo_out}")
print(f"Saved tabular Parquet: {tabular_out}")
print("\nCounts by category:")
print(pois["poi_category"].value_counts())

Fetching food_drink ...
Fetching train_station ...
Fetching shop ...
Fetching landmark ...
Saved GeoParquet: /Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/bronze_osm_chicago_pois.geoparquet
Saved tabular Parquet: /Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/bronze_osm_chicago_pois.parquet

Counts by category:
poi_category
food_drink       5791
shop             1783
landmark          973
train_station     570
Name: count, dtype: int64


## Create Parquet File for efficient Data Handling

Loop over the raw-data folder and create bronze Parquet files from CSV inputs and the compressed sample fallback. A file is regenerated automatically when its raw source is newer; the `overwrite` flag can force regeneration. A short row-count check confirms each conversion.

In [ ]:
overwrite = False

raw_dir = RAW_DIR
processed_dir = PROCESSED_DIR

if not raw_dir.exists():
    raise FileNotFoundError(f"Raw data folder does not exist: {raw_dir}")

processed_dir.mkdir(parents=True, exist_ok=True)

raw_files = sorted(raw_dir.glob("*.csv"))
if PATHS.raw_taxi_trips.suffix == ".parquet":
    raw_files.append(PATHS.raw_taxi_trips)

if not raw_files:
    print(f"No raw CSV or Parquet files found in {raw_dir}")

for raw_path in raw_files:
    parquet_path = processed_dir / f"bronze_{raw_path.stem}.parquet"

    output_is_current = (
        parquet_path.exists()
        and parquet_path.stat().st_mtime >= raw_path.stat().st_mtime
    )
    if output_is_current and not overwrite:
        print(f"Skipping {raw_path.name}: {parquet_path.name} is current")
        
        print("Quality check - Rows in parquet file:")
    
        duckdb.sql(f"""
        SELECT count(*)
        FROM read_parquet('{parquet_path}')
        """).show()
    
        continue

    print(f"Converting {raw_path.name} -> {parquet_path.name}")
    source_scan = (
        f"read_parquet('{raw_path}')"
        if raw_path.suffix == ".parquet"
        else f"read_csv_auto('{raw_path}', header=true, sample_size=-1)"
    )
    duckdb.sql(f"""
        COPY (SELECT * FROM {source_scan})
        TO '{parquet_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)
    
    print("Quality check - Rows in parquet file:")
    
    duckdb.sql(f"""
    SELECT count(*)
    FROM read_parquet('{parquet_path}')
    """).show()



print("Done creating bronze Parquet files from raw inputs.")